**Exploritory Data Analysis**

In [26]:
import pandas as pd


""" --- Exploratory Data Analysis --- """
df = pd.read_csv("Data/IMDB.csv")

# Sample Head of dataset
print(df.head(5))
print(df.tail(5))

# Dataset shape and Description
print("\n",df.shape)
print("\n --- Dataset Description --- ")
print(df.describe())
print("-----------------------------")

# Dataset information
print("\n --- Dataset Summary --- ")
print(df.info())
print("-----------------------------")

# Dataset Null Values
print("\n --- Missing Values --- ")
print(df.isnull().sum())
print("-----------------------------")

# Dataset Duplicates
print("\n --- Duplicate Values --- ")
print(df[df.duplicated(subset=['review'])])
print("-----------------------------")

""" --- Initial Cleaning before EDA ---"""
# Droping Duplicates
df = df.drop_duplicates(subset=['review'])
print(df.shape)



                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
                                                  review sentiment
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative

 (50000, 2)

 --- Dataset Description --- 
                                                   review sentiment
count                                               50000     50000
unique                                  

In [30]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

''' --- CONFIGURATIONS --- '''
OUTPUT_DIR = "eda_outputs"
TOP_N_WORDS = 25
TOP_N_BIGRAMS = 20

sns.set_style("whitegrid")
os.makedirs(OUTPUT_DIR, exist_ok=True)
STOPWORDS = set(stopwords.words("english"))

POS_COLOR, NEG_COLOR = "#2ecc71", "#e74c3c"

HTML_TAG_RE = re.compile(r"<[^>]+>")
URL_RE = re.compile(r"http\S+|www\.\S+")
NON_ALPHA_RE = re.compile(r"[^a-zA-Z\s]")
ALLCAPS_WORD_RE = re.compile(r"\b[A-Z]{2,}\b")

def save_fig(fig, name):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, bbox_inches="tight")
    print(f"  -> saved {path}")
    plt.close(fig)


def clean_text(text):
    """Lowercase, strip HTML/URLs/punctuation/digits ."""
    text = HTML_TAG_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = text.lower()
    text = NON_ALPHA_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text, min_len=3, remove_stopwords=True):
    tokens = clean_text(text).split()
    if remove_stopwords:
        tokens = [t for t in tokens if t not in STOPWORDS and len(t) >= min_len]
    return tokens

''' --- CLASS IMBALANCE --- '''

print("\n"+"=" * 70)
print(" --- CLASS IMBALANCE --- ")
print("=" * 70)
counts = df["sentiment"].value_counts()
pct = (counts / counts.sum() * 100).round(2)
print(counts)
print(f"Percentages:\n{pct}")

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.barplot(
    x=counts.index, y=counts.values,
    hue=counts.index, legend=False,
    palette={"positive": POS_COLOR, "negative": NEG_COLOR}, ax=ax,
)
for i, v in enumerate(counts.values):
    ax.text(i, v + max(counts.values) * 0.01, f"{v:,} ({pct.iloc[i]}%)",
            ha="center", fontweight="bold")
ax.set_title("Class Distribution: Positive vs Negative Reviews", fontsize=13, fontweight="bold")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Number of Reviews")
save_fig(fig, "class_balance.png")




 --- CLASS IMBALANCE --- 
sentiment
positive    24884
negative    24698
Name: count, dtype: int64
Percentages:
sentiment
positive    50.19
negative    49.81
Name: count, dtype: float64
  -> saved eda_outputs\01_class_balance.png


In [32]:

''' --- WORD COUNT PER REVIEW --- '''

print("\n" + "=" * 70)
print(" WORD COUNT DISTRIBUTION")
print("=" * 70)
df["word_count"] = df["review"].str.split().apply(len)
print(df["word_count"].describe())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df, x="word_count", hue="sentiment", bins=50, kde=True,
             palette={"positive": POS_COLOR, "negative": NEG_COLOR}, ax=axes[0])
axes[0].set_title("Distribution of Review Word Counts")
axes[0].set_xlabel("Word Count")
axes[0].set_xlim(0, df["word_count"].quantile(0.99))

sns.boxplot(data=df, x="sentiment", y="word_count",
            hue="sentiment", legend=False,
            palette={"positive": POS_COLOR, "negative": NEG_COLOR}, ax=axes[1])
axes[1].set_title("Word Count by Sentiment (Boxplot)")
axes[1].set_ylim(0, df["word_count"].quantile(0.99))
save_fig(fig, "word_count_dist.png")



 WORD COUNT DISTRIBUTION
count    49582.000000
mean       231.350167
std        171.542020
min          4.000000
25%        126.000000
50%        173.000000
75%        281.000000
max       2470.000000
Name: word_count, dtype: float64
  -> saved eda_outputs\word_count_dist.png


In [34]:

''' --- NOISE IDENTIFICATION --- '''

print("\n" + "=" * 70)
print(" NOISE IDENTIFICATION")
print("=" * 70)
has_url = df["review"].str.contains(URL_RE, regex=True)
has_html = df["review"].str.contains(HTML_TAG_RE, regex=True)
has_digit = df["review"].str.contains(r"\d", regex=True)
has_special = df["review"].str.contains(r"[^a-zA-Z0-9\s\.\,\!\?\'\"]", regex=True)
has_allcaps = df["review"].str.contains(ALLCAPS_WORD_RE, regex=True)

noise_summary = pd.Series({
    "Contains URL": has_url.sum(),
    "Contains HTML tags": has_html.sum(),
    "Contains digits": has_digit.sum(),
    "Contains special chars": has_special.sum(),
    "Contains ALL-CAPS word": has_allcaps.sum(),
})
print(noise_summary)
print(f"\nExample with HTML tags:\n{df.loc[has_html, 'review'].iloc[0][:200]}...")
if has_url.any():
    print(f"\nExample with URL:\n{df.loc[has_url, 'review'].iloc[0][:200]}...")

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(x=noise_summary.values, y=noise_summary.index, hue=noise_summary.index,
            legend=False, palette="rocket", ax=ax)
for i, v in enumerate(noise_summary.values):
    ax.text(v + 300, i, f"{v:,} ({v/len(df)*100:.1f}%)", va="center")
ax.set_title("Noise Patterns Found in Raw Reviews", fontsize=13, fontweight="bold")
ax.set_xlabel("Number of Reviews Affected")
save_fig(fig, "noise_summary.png")


 NOISE IDENTIFICATION
Contains URL                196
Contains HTML tags        28968
Contains digits           27786
Contains special chars    44635
Contains ALL-CAPS word    23982
dtype: int64

Example with HTML tags:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo...

Example with URL:
Mario Lewis of the Competitive Enterprise Institute has written a definitive 120-page point-by-point, line-by-line refutation of this mendacious film, which should be titled A CONVENIENT LIE. The webs...
  -> saved eda_outputs\noise_summary.png


In [35]:
from collections import Counter

''' --- MOST FREQUENT WORDS --- '''

print("\n" + "=" * 70)
print(" MOST FREQUENT WORDS")
print("=" * 70)
all_tokens = [tok for review in df["review"] for tok in tokenize(review)]
word_freq = Counter(all_tokens)
top_words = word_freq.most_common(TOP_N_WORDS)
print(f"Total tokens : {len(all_tokens):,} | Unique words: {len(word_freq):,}")
print(f"Top {TOP_N_WORDS}: {top_words}")

words, freqs = zip(*top_words)
fig, ax = plt.subplots(figsize=(8, 9))
sns.barplot(x=list(freqs), y=list(words), hue=list(words), legend=False,
            palette="viridis", ax=ax)
ax.set_title(f"Top {TOP_N_WORDS} Most Frequent Words (Whole Dataset)", fontsize=13, fontweight="bold")
ax.set_xlabel("Frequency")
save_fig(fig, "top_words.png")


 MOST FREQUENT WORDS
Total tokens : 5,785,524 | Unique words: 98,632
Top 25: [('movie', 87276), ('film', 79148), ('one', 53205), ('like', 39843), ('good', 29547), ('time', 24904), ('even', 24669), ('would', 24401), ('story', 22973), ('really', 22924), ('see', 22823), ('well', 21108), ('much', 19153), ('bad', 18292), ('get', 18275), ('people', 18021), ('great', 18019), ('also', 17851), ('first', 17432), ('made', 16004), ('make', 15781), ('way', 15517), ('could', 15420), ('movies', 15224), ('characters', 14321)]
  -> saved eda_outputs\top_words.png
